# Notebook 55: STRAT-004 Relaxed Entry Test (2 of 3 Conditions)

**Finding from Notebook 54:**
- ALL 3 conditions: 7.5% of bars
- 2 of 3 conditions: 19.5% of bars (2.6x more opportunities)

**Goal:** Test if relaxing to 2-of-3 conditions increases trades while maintaining edge.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
from pathlib import Path

DATA_DIR = Path("../data")
HOURLY_DIR = DATA_DIR / "hourly"

In [ ]:
# Load data
def load_hourly():
    price = pd.read_parquet(HOURLY_DIR / "price.parquet")
    sopr = pd.read_parquet(HOURLY_DIR / "sopr.parquet")
    sopr_sth = pd.read_parquet(HOURLY_DIR / "sopr_sth.parquet")
    realized_loss = pd.read_parquet(HOURLY_DIR / "realized_loss.parquet")
    
    df = price.rename(columns={"value": "price"}).set_index("time")
    df["sopr"] = sopr.set_index("time")["value"]
    df["sopr_sth"] = sopr_sth.set_index("time")["value"]
    df["realized_loss"] = realized_loss.set_index("time")["value"]
    return df

def add_zscore(df, window):
    df = df.copy()
    df["rl_mean"] = df["realized_loss"].rolling(window=window, min_periods=window//2).mean()
    df["rl_std"] = df["realized_loss"].rolling(window=window, min_periods=window//2).std()
    df["rl_zscore"] = (df["realized_loss"] - df["rl_mean"]) / df["rl_std"]
    return df

df_full = load_hourly()
df_full = add_zscore(df_full, 365 * 24)

# Filter to backtest period
START = "2019-01-01"
df = df_full[df_full.index >= START].dropna()
years = (df.index.max() - df.index.min()).days / 365.25

print(f"Data: {len(df):,} hourly bars ({years:.1f} years)")

In [ ]:
def get_metrics(pf, years):
    trades = pf.trades.records_readable
    if len(trades) == 0:
        return None
    
    total_return = pf.total_return() * 100
    durations = (trades["Exit Timestamp"] - trades["Entry Timestamp"]).dropna()
    avg_days = durations.mean().total_seconds() / 86400 if len(durations) > 0 else 0
    
    winning = trades[trades["PnL"] > 0]
    losing = trades[trades["PnL"] < 0]
    
    return {
        "return": total_return,
        "cagr": ((1 + total_return/100) ** (1/years) - 1) * 100,
        "sharpe": pf.sharpe_ratio(),
        "max_dd": pf.max_drawdown() * 100,
        "trades": len(trades),
        "trades_yr": len(trades) / years,
        "avg_days": avg_days,
        "win_rate": (trades["PnL"] > 0).mean() * 100,
        "profit_factor": abs(winning["PnL"].sum() / losing["PnL"].sum()) if len(losing) > 0 and losing["PnL"].sum() != 0 else np.inf,
    }

## 1. Define Entry Conditions

In [ ]:
# Individual conditions
c1 = df["sopr"] < 1           # SOPR < 1
c2 = df["sopr_sth"] < 1       # STH-SOPR < 1  
c3 = df["rl_zscore"] > 0.5    # Realized Loss z-score > 0.5

# Entry variations
entry_all3 = c1 & c2 & c3     # Original: ALL 3 conditions

# 2-of-3 combinations
entry_c1_c2 = c1 & c2         # SOPR + STH-SOPR (no RL filter)
entry_c1_c3 = c1 & c3         # SOPR + RL (no STH filter)
entry_c2_c3 = c2 & c3         # STH-SOPR + RL (no SOPR filter)

# Any 2 of 3
entry_any2 = (c1.astype(int) + c2.astype(int) + c3.astype(int)) >= 2

print("Entry condition frequency:")
print(f"  ALL 3 (original):   {entry_all3.mean()*100:5.1f}%")
print(f"  C1+C2 (SOPR+STH):   {entry_c1_c2.mean()*100:5.1f}%")
print(f"  C1+C3 (SOPR+RL):    {entry_c1_c3.mean()*100:5.1f}%")
print(f"  C2+C3 (STH+RL):     {entry_c2_c3.mean()*100:5.1f}%")
print(f"  ANY 2 of 3:         {entry_any2.mean()*100:5.1f}%")

## 2. Test All Entry Variations

In [ ]:
def run_backtest(df, entry_cond, trail=0.12):
    entry = entry_cond & ~entry_cond.shift(1).fillna(False)
    
    if entry.sum() == 0:
        return None
    
    return vbt.Portfolio.from_signals(
        close=df["price"],
        entries=entry,
        exits=None,
        sl_stop=trail,
        sl_trail=True,
        freq="1h",
        init_cash=10000,
        fees=0.001
    )

# Test all variations
variations = [
    ("ALL 3 (Original)", entry_all3),
    ("SOPR + STH-SOPR", entry_c1_c2),
    ("SOPR + RL z-score", entry_c1_c3),
    ("STH-SOPR + RL z-score", entry_c2_c3),
    ("ANY 2 of 3", entry_any2),
]

print("="*120)
print("ENTRY CONDITION COMPARISON (1H @ 12% Trail)")
print("="*120)
print(f"\n{'Entry Condition':<25} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'Days':>8} {'Win%':>8}")
print("-"*120)

results = {}
for name, cond in variations:
    pf = run_backtest(df, cond)
    if pf:
        m = get_metrics(pf, years)
        results[name] = {"m": m, "pf": pf}
        print(f"{name:<25} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['avg_days']:>8.1f} {m['win_rate']:>7.0f}%")

## 3. Forward Test (Past Year) - All Variations

In [ ]:
# Forward test period
FORWARD_START = "2024-01-15"
FORWARD_END = "2025-01-15"

df_fwd = df[(df.index >= FORWARD_START) & (df.index <= FORWARD_END)]
fwd_years = (df_fwd.index.max() - df_fwd.index.min()).days / 365.25

# Re-calculate conditions for forward period
c1_fwd = df_fwd["sopr"] < 1
c2_fwd = df_fwd["sopr_sth"] < 1
c3_fwd = df_fwd["rl_zscore"] > 0.5

variations_fwd = [
    ("ALL 3 (Original)", c1_fwd & c2_fwd & c3_fwd),
    ("SOPR + STH-SOPR", c1_fwd & c2_fwd),
    ("SOPR + RL z-score", c1_fwd & c3_fwd),
    ("STH-SOPR + RL z-score", c2_fwd & c3_fwd),
    ("ANY 2 of 3", (c1_fwd.astype(int) + c2_fwd.astype(int) + c3_fwd.astype(int)) >= 2),
]

bh_return = (df_fwd["price"].iloc[-1] / df_fwd["price"].iloc[0] - 1) * 100

print("\n" + "="*120)
print(f"FORWARD TEST: {FORWARD_START} to {FORWARD_END} (B&H: {bh_return:+.1f}%)")
print("="*120)
print(f"\n{'Entry Condition':<25} {'Return':>10} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8} {'Tr/Yr':>8} {'Days':>8} {'Win%':>8}")
print("-"*120)

fwd_results = {}
for name, cond in variations_fwd:
    pf = run_backtest(df_fwd, cond)
    if pf:
        m = get_metrics(pf, fwd_years)
        fwd_results[name] = {"m": m, "pf": pf}
        print(f"{name:<25} {m['return']:>+9.0f}% {m['cagr']:>+7.1f}% {m['sharpe']:>8.2f} {m['max_dd']:>7.1f}% {m['trades']:>8} {m['trades_yr']:>8.1f} {m['avg_days']:>8.1f} {m['win_rate']:>7.0f}%")
    else:
        print(f"{name:<25} {'No trades':>10}")

## 4. Detailed Comparison: Original vs Best 2-of-3

In [ ]:
print("\n" + "="*80)
print("DETAILED COMPARISON")
print("="*80)

# Find best 2-of-3 variant by return
two_of_3_options = ["SOPR + STH-SOPR", "SOPR + RL z-score", "STH-SOPR + RL z-score", "ANY 2 of 3"]
best_2of3 = max(two_of_3_options, key=lambda x: results.get(x, {}).get("m", {}).get("return", -999))

orig = results["ALL 3 (Original)"]["m"]
relaxed = results[best_2of3]["m"]

print(f"\n{'Metric':<20} {'ALL 3 (Original)':>18} {best_2of3:>25} {'Difference':>15}")
print("-"*80)

comparisons = [
    ("Total Return", "return", "%"),
    ("CAGR", "cagr", "%"),
    ("Sharpe", "sharpe", ""),
    ("Max Drawdown", "max_dd", "%"),
    ("Trades", "trades", ""),
    ("Trades/Year", "trades_yr", ""),
    ("Avg Hold Days", "avg_days", ""),
    ("Win Rate", "win_rate", "%"),
]

for name, key, suffix in comparisons:
    o = orig[key]
    r = relaxed[key]
    diff = r - o
    print(f"{name:<20} {o:>17.1f}{suffix} {r:>24.1f}{suffix} {diff:>+14.1f}{suffix}")

## 5. Equity Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['blue', 'green', 'orange', 'red', 'purple']
for i, (name, data) in enumerate(results.items()):
    pf = data["pf"]
    m = data["m"]
    equity = pf.value().resample('D').last()
    equity.plot(ax=ax, label=f"{name}: {m['return']:+,.0f}% ({m['trades']} trades)", 
                color=colors[i], linewidth=2 if name == "ALL 3 (Original)" else 1.5,
                linestyle='-' if name == "ALL 3 (Original)" else '--')

ax.set_title("Entry Condition Comparison: STRAT-004 Variations")
ax.set_ylabel("Portfolio Value ($)")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 6. Trade Details for Each Variation

In [ ]:
print("\n" + "="*80)
print("TRADE COUNT BY YEAR")
print("="*80)

for name, data in results.items():
    trades = data["pf"].trades.records_readable
    trades["Year"] = trades["Entry Timestamp"].dt.year
    yearly = trades.groupby("Year").size()
    
    print(f"\n{name}:")
    for year, count in yearly.items():
        print(f"  {year}: {count} trades")

## 7. Recommendation

In [ ]:
print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

# Score each variation
print("\nScoring (higher is better):")
print(f"{'Variation':<25} {'Return':>10} {'Sharpe':>10} {'Trades':>10} {'Score':>10}")
print("-"*70)

scores = []
for name, data in results.items():
    m = data["m"]
    # Score = CAGR * Sharpe * sqrt(trades_per_year)
    score = m["cagr"] * max(m["sharpe"], 0) * np.sqrt(m["trades_yr"])
    scores.append((name, score, m))
    print(f"{name:<25} {m['return']:>+9.0f}% {m['sharpe']:>10.2f} {m['trades']:>10} {score:>10.1f}")

scores.sort(key=lambda x: x[1], reverse=True)
best_name, best_score, best_m = scores[0]

print(f"\n{'='*60}")
print(f"BEST OVERALL: {best_name}")
print(f"{'='*60}")
print(f"  Return:     {best_m['return']:+,.0f}%")
print(f"  CAGR:       {best_m['cagr']:+.1f}%")
print(f"  Sharpe:     {best_m['sharpe']:.2f}")
print(f"  Trades/Yr:  {best_m['trades_yr']:.1f}")
print(f"  Win Rate:   {best_m['win_rate']:.0f}%")

In [ ]:
print("\n" + "="*80)
print("FINAL STRATEGY VARIANTS")
print("="*80)

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║  STRAT-004a: STRICT (Original)                                               ║
║  Entry: SOPR < 1 AND STH-SOPR < 1 AND RL z-score > 0.5                       ║
║  Use when: Want highest quality signals, fewer trades                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════════════════════╗
║  STRAT-004b: RELAXED (2 of 3)                                                ║
║  Entry: Any 2 of: SOPR < 1, STH-SOPR < 1, RL z-score > 0.5                  ║
║  Use when: Want more trades, accept slightly lower quality                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")